## Data Ingestion — Aviationstack flights (Bronze)

Loads the day's **landed departures from the tracked airports** (Kuala Lumpur KUL and
Penang PEN) from the [Aviationstack](https://aviationstack.com) API into
`workspace.default.bronze_flight`. These flights feed the delay insights.

```text
Aviationstack API  →  JSON  →  Spark DataFrame  →  Delta table (bronze_flight)
```

**Why landed flights only:** a flight's departure delay is only final once it has left.
Scheduled flights have no delay yet, and flights still in the air may change.

**Working within the free plan (about 100 calls a month):**
- One call per airport per night, run by the *Flights* job at 23:30 Malaysia time.
- A call returns at most 100 flights, **earliest first**. So each night starts at a different
  point in the day (a rotating offset: 0, 100, 200, 300, then 0 again), and every hour is
  covered over a few nights. If the offset runs past the day's last flight, the call is
  repeated from the start, which costs one extra call.

### What gets cleaned

Nothing yet: Bronze keeps the data as Aviationstack sends it, and `02b_transform_silver_flights`
does the cleaning. This notebook only:

| Step | What it does |
|---|---|
| **Flatten** | Turns each nested JSON flight into one flat row, one per flight number |
| **Explicit schema** | Delays as whole numbers; everything else, including times, as text (converted in Silver) |
| **Missing sections** | A flight without e.g. a `codeshared` block gets nulls, rather than failing |

### Known limitations

- **A sample, not every flight:** at most 100 flight numbers per airport per night. Aviationstack
  counts every codeshare number separately, so after Silver removes the copies a night's sample
  is well under 100 real flights.
- **Late departures are missed:** the job runs at 23:30, and flights that haven't landed by then
  aren't "landed" yet, so the last flights of the day are under-represented.
- **Extra calls for Penang:** Penang's total is smaller than KUL's. If it is under 300, the higher
  offsets run past its last flight on some nights and need a second call. The printed totals show
  whether this happens.
- **Local times labelled UTC:** times are the airport's local time marked `+00:00`. They're kept
  as-is here; Gold reads them in UTC so the clock time stays right.
- **No flights stops the run:** if neither airport returns flights, the notebook stops at the
  sample-record cell and `bronze_flight` keeps the previous night's data.


In [ ]:
import requests
from pyspark.sql.types import *

# The Aviationstack key is stored as a Databricks secret, never in the notebook.
API_KEY = dbutils.secrets.get(scope="flightpulse", key="aviationstack_key")

url = "https://api.aviationstack.com/v1/flights"

# Airports tracked for delay insights. Each costs one call a day, and the free
# plan allows about 100 calls a month, so keep this list short.
TRACKED_AIRPORTS = ["KUL", "PEN"]

# Landed flights only: their departure delay is final. A call returns at most
# 100 flights, earliest first, and KUL has several hundred a day, so each day
# starts at a different point (offset) to cover every hour of the day over a
# week or so. Smaller airports fit in one page; if the offset runs past the end,
# the call is repeated from the start (one extra call).
import datetime

PAGE_SIZE = 100
PAGES_TO_ROTATE = 4  # offsets 0, 100, 200, 300, then back to 0
day_number = datetime.date.today().toordinal()

all_flights = []

for airport in TRACKED_AIRPORTS:
    offset = (day_number % PAGES_TO_ROTATE) * PAGE_SIZE

    def fetch(offset):
        response = requests.get(
            url,
            params={
                "access_key": API_KEY,
                "dep_iata": airport,
                "flight_status": "landed",
                "limit": PAGE_SIZE,
                "offset": offset
            },
            timeout=30
        )
        response.raise_for_status()
        return response.json()

    page = fetch(offset)
    if not page["data"] and offset > 0:
        offset = 0
        page = fetch(offset)

    batch = page["data"]
    all_flights.extend(batch)

    # Which part of the day did the sample cover? (Times are the airport's local time.)
    times = sorted(
        (f.get("departure") or {}).get("scheduled") or ""
        for f in batch
        if (f.get("departure") or {}).get("scheduled")
    )
    total = (page.get("pagination") or {}).get("total")
    print(f"{airport}: {len(batch)} of {total} flights (offset {offset}), scheduled {times[0][:16] if times else '-'} to {times[-1][:16] if times else '-'}")

data = {"data": all_flights}

delayed = sum(
    1 for f in all_flights
    if ((f.get("departure") or {}).get("delay") or 0) > 15
)

print("Flights received:", len(all_flights))
print("Departed more than 15 minutes late:", delayed)


A quick look at one raw record, to check the shape of the response. The fields used
below are under `departure`, `arrival`, `airline` and `flight`.

In [ ]:
flights = data["data"]
print(flights[0])

### Schema

An explicit schema keeps column types stable from run to run, instead of letting Spark
guess them from the data.

The `codeshared_*` columns matter: when one plane is sold under several airlines'
flight numbers, Aviationstack returns a row per flight number, and the extra rows name
the airline that actually operates the plane. Silver uses them to count each flight once.

In [ ]:
from pyspark.sql.types import *

flights = data["data"]
schema = StructType([
      StructField("flight_date", StringType(), True),
      StructField("flight_status", StringType(), True),

      StructField("departure_airport", StringType(), True),
      StructField("departure_timezone", StringType(), True),
      StructField("departure_iata", StringType(), True),
      StructField("departure_icao", StringType(), True),
      StructField("departure_terminal", StringType(), True),
      StructField("departure_gate", StringType(), True),
      StructField("departure_delay", IntegerType(), True),
      StructField("departure_scheduled", StringType(), True),
      StructField("departure_estimated", StringType(), True),
      StructField("departure_actual", StringType(), True),

      StructField("arrival_airport", StringType(), True),
      StructField("arrival_timezone", StringType(), True),
      StructField("arrival_iata", StringType(), True),
      StructField("arrival_icao", StringType(), True),
      StructField("arrival_terminal", StringType(), True),
      StructField("arrival_baggage", StringType(), True),
      StructField("arrival_scheduled", StringType(), True),
      StructField("arrival_delay", IntegerType(), True),
      StructField("arrival_estimated", StringType(), True),
      StructField("arrival_actual", StringType(), True),

      StructField("airline_name", StringType(), True),
      StructField("airline_iata", StringType(), True),
      StructField("airline_icao", StringType(), True),

      StructField("flight_number", StringType(), True),
      StructField("flight_iata", StringType(), True),
      StructField("flight_icao", StringType(), True),

      StructField("codeshared_airline_name", StringType(), True),
      StructField("codeshared_airline_iata", StringType(), True),
      StructField("codeshared_airline_icao", StringType(), True),
      StructField("codeshared_flight_number", StringType(), True),
      StructField("codeshared_flight_iata", StringType(), True),
      StructField("codeshared_flight_icao", StringType(), True)
  ])

### Flatten the response

Each flight arrives as nested JSON. This turns it into one flat row per flight, in the
same order as the schema. Missing sections (e.g. no `codeshared` block) become nulls.

In [ ]:
rows = []

for flight in flights:

      departure = flight.get("departure") or {}
      arrival = flight.get("arrival") or {}
      airline = flight.get("airline") or {}
      flight_info = flight.get("flight") or {}
      codeshared = flight_info.get("codeshared") or {}

      rows.append([
          flight.get("flight_date"),
          flight.get("flight_status"),

          departure.get("airport"),
          departure.get("timezone"),
          departure.get("iata"),
          departure.get("icao"),
          departure.get("terminal"),
          departure.get("gate"),
          departure.get("delay"),
          departure.get("scheduled"),
          departure.get("estimated"),
          departure.get("actual"),

          arrival.get("airport"),
          arrival.get("timezone"),
          arrival.get("iata"),
          arrival.get("icao"),
          arrival.get("terminal"),
          arrival.get("baggage"),
          arrival.get("scheduled"),
          arrival.get("delay"),
          arrival.get("estimated"),
          arrival.get("actual"),

          airline.get("name"),
          airline.get("iata"),
          airline.get("icao"),

          flight_info.get("number"),
          flight_info.get("iata"),
          flight_info.get("icao"),

          codeshared.get("airline_name"),
          codeshared.get("airline_iata"),
          codeshared.get("airline_icao"),
          codeshared.get("flight_number"),
          codeshared.get("flight_iata"),
          codeshared.get("flight_icao")
      ])

### Create the DataFrame

In [ ]:
spark_df = spark.createDataFrame(rows, schema)
display(spark_df)


### Save to Bronze

Bronze holds the latest fetch only (`overwrite`). History is kept one layer up, in
`silver_flights_history`, where each day's flights are merged in.
`overwriteSchema` lets the table pick up new columns if the schema changes.

In [ ]:
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.bronze_flight")


### Check the result

In [ ]:
display(
    spark.table("workspace.default.bronze_flight")
)